# Binder Design Pipeline (Colab)
RFD3 → MPNN → RF3

This notebook runs the protein binder design pipeline entirely on Google Colab. It is a translation of the DTU HPC version; the computational steps are identical but job submission (LSF/bsub) is replaced by direct in-notebook execution.

**Target**: CLR:RAMP1 heterodimeric class-B GPCR complex — designing antagonistic binders that engage the receptor and block ligand-induced activation.

## Step 0: Colab Setup

Run this cell first, every session. It:
1. Mounts your Google Drive
2. Installs required Python packages
3. Clones LigandMPNN (small, public)
4. Adds support scripts to the Python path

> **One-time instructor setup**: See `CHECKPOINT.md` for the required Drive folder structure and which model weights to upload.

In [ ]:
# ── Python version check (rc-foundry requires 3.12) ──────────────────────────
import sys
assert sys.version_info >= (3, 12), (
    f'This notebook requires Python 3.12+. You have {sys.version}. '
    'In Colab: Runtime → Change runtime type → Python 3.12'
)
print(f'Python {sys.version_info.major}.{sys.version_info.minor} ✓')

# ── Mount Drive ──────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Drive paths (don't change unless necessary. Paths hardcoded in RF3 .json) ─────────────────────────
DRIVE_BASE     = '/content/drive/MyDrive/protein_design'
MODELS_DIR     = f'{DRIVE_BASE}/models'
SCRIPTS_DIR    = f'{DRIVE_BASE}/support'
INPUTS_DIR     = f'{DRIVE_BASE}/inputs'
LIGANDMPNN_DIR = f'{DRIVE_BASE}/LigandMPNN'

import subprocess, glob as _glob, os, re

# ── Environment flags (inherited by all subprocesses) ─────────────────────────
os.environ['WANDB_DISABLED']            = 'true'      # suppress wandb login prompts
os.environ['WANDB_MODE']               = 'disabled'
os.environ['PYTORCH_CUDA_ALLOC_CONF']  = 'expandable_segments:True'  # reduce OOM risk

# ── Install rc-foundry (provides rfd3 + rf3 + mpnn CLIs) ─────────────────────
wheel = _glob.glob(f'{MODELS_DIR}/rc_foundry-*.whl')
if wheel:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', wheel[0]], check=True)
    print(f'Installed {os.path.basename(wheel[0])}')
else:
    print('Wheel not found on Drive — installing from GitHub (slower)...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'rc-foundry[all] @ git+https://github.com/RosettaCommons/foundry.git'],
        check=True
    )

# ── Reinstall torchvision matched to the installed torch + CUDA version ───────
import torch
cuda_tag = ('cu' + torch.version.cuda.replace('.', '')) if torch.cuda.is_available() else 'cpu'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchvision',
    '--index-url', f'https://download.pytorch.org/whl/{cuda_tag}'], check=True)
print(f'torchvision reinstalled for torch {torch.__version__} / {cuda_tag}')

# ── Install remaining dependencies ────────────────────────────────────────────
def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

_pip('biopython', 'numpy', 'pandas', 'scipy', 'matplotlib', 'seaborn',
     'prody',            # LigandMPNN
     'ml-collections',   # LigandMPNN / OpenFold
     'dm-tree')          # LigandMPNN / OpenFold

# ── Clone LigandMPNN (public, small) ─────────────────────────────────────────
if not os.path.exists(LIGANDMPNN_DIR):
    subprocess.run(
        ['git', 'clone', 'https://github.com/dauparas/LigandMPNN.git', LIGANDMPNN_DIR],
        check=True
    )
    print('LigandMPNN cloned.')
else:
    print('LigandMPNN already present.')

# ── Patch LigandMPNN / OpenFold for NumPy >= 1.24 compatibility ──────────────
_DEPRECATED = re.compile(r'\bnp\.(int|float|bool|complex|str)\b(?![\d_])')
_patched = 0
for _root, _dirs, _files in os.walk(f'{LIGANDMPNN_DIR}/openfold'):
    for _fname in _files:
        if not _fname.endswith('.py'):
            continue
        _path = os.path.join(_root, _fname)
        _text = open(_path).read()
        _new  = _DEPRECATED.sub(lambda m: m.group(1), _text)
        if _new != _text:
            open(_path, 'w').write(_new)
            _patched += 1
print(f'NumPy compat patch applied to {_patched} file(s) in LigandMPNN/openfold.')

# ── Add support scripts to path ───────────────────────────────────────────────
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

print('Setup complete.')

## Imports and directories

In [ ]:
# --- Core ---
import os, sys, re, glob, json, math, time, ast, stat, shutil, random, string, tempfile, warnings, subprocess, csv, shlex
import itertools
from pathlib import Path
from collections import Counter, defaultdict, OrderedDict
from datetime import datetime
from difflib import SequenceMatcher
from copy import deepcopy

# --- Data / numerics ---
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

# --- Plotting ---
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import to_rgba, ListedColormap, LinearSegmentedColormap
import seaborn as sns

# --- BioPython ---
from Bio import SeqIO, PDB
from Bio.PDB import PDBParser, PPBuilder

# --- Notebook display ---
from IPython.display import display, HTML

# --- Shared utilities (loaded from Drive via SCRIPTS_DIR in sys.path) ---
import rf3_metrics  # provides gather_rf3_metrics

In [ ]:
# ── Helper: stream a shell command's output in the notebook ───────────────────
def run_step(cmd, label=''):
    """Run a shell command, printing output line by line."""
    if label:
        print(f'\n{"-"*60}\n{label}\n{"-"*60}')
    print(f'$ {cmd}\n')
    proc = subprocess.Popen(cmd, shell=True, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed (exit {proc.returncode})')

In [ ]:
# ── Campaign settings ─────────────────────────────────────────────────────────
student_name = 'student'         # ← CHANGE THIS to your name
campaign     = 'CLR_RAMP1'
experiment   = 'exp_01'
display_all  = True

# Outputs go to YOUR OWN Drive (not the shared folder — you can't write there).
# Everything else (models, scripts, inputs) reads from DRIVE_BASE above.
STUDENT_BASE = '/content/drive/MyDrive'
working_dir  = f'{STUDENT_BASE}/binder_design_runs/{student_name}/{campaign}/{experiment}'
os.makedirs(working_dir, exist_ok=True)
os.chdir(working_dir)

def setup_directories(base_dir, dirs_list):
    result = {}
    for dir_name in dirs_list:
        full_path = os.path.join(base_dir, dir_name)
        os.makedirs(full_path, exist_ok=True)
        result[dir_name + '_dir'] = full_path
    return result

wrk_dirs_list = ['cmds', 'logs', 'cute_pics', 'configs', 'scores', 'inputs',
                 'diffusion_out', 'rf3_out', 'mpnn_out']
_dirs = setup_directories(working_dir, wrk_dirs_list)

cmds_dir          = _dirs['cmds_dir']
logs_dir          = _dirs['logs_dir']
cute_pics_dir     = _dirs['cute_pics_dir']
configs_dir       = _dirs['configs_dir']
scores_dir        = _dirs['scores_dir']
inputs_dir        = _dirs['inputs_dir']
diffusion_out_dir = _dirs['diffusion_out_dir']
rf3_out_dir       = _dirs['rf3_out_dir']
mpnn_out_dir      = _dirs['mpnn_out_dir']

# Copy input PDB from shared Drive inputs into working inputs dir
for fname in ['6e3y_cut.pdb', '6e3y_binding_site.pdb']:
    src = f'{INPUTS_DIR}/{fname}'
    dst = f'{inputs_dir}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        import shutil; shutil.copy2(src, dst)

if display_all:
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', None)

print(f'Campaign : {campaign} / {experiment}')
print(f'Directory: {working_dir}')

## RFD3

RFdiffusion3 (RFD3) generates de novo binder backbone structures by diffusing from noise while steering the design toward your target residues. Hotspots guide where the binder contacts the target; the contig defines how many residues to design and which target chains to include.

Full parameter reference: [RFD3 protein binder design docs](https://github.com/RosettaCommons/foundry/blob/production/models/rfd3/docs/protein_binder_design.md)

Key parameters:
- **`contig`**: free range (e.g. `50-150`) sets binder length; fixed segments like `B33-54` copy target residues from the input PDB. `/0` separates chains.
- **`select_hotspots`**: atom-level target residues the binder should contact.
- **`infer_ori_strategy`**: `"hotspots"` places the binder origin relative to the hotspot centroid.

### Configure RFD3

In [ ]:
design_name = 'CLR_RAMP1_test'
dialect     = 2
input_pdb   = f'{inputs_dir}/6e3y_cut.pdb'

# Binder length (free range) + fixed target residue segments
contig = '50-150,/0,B33-54,10,B64-106,3,B110-200,/0,C29-120'
length = '298-398'

redesign_motif_sidechains = False
json_name = 'CLR_RAMP1_rfd3_test.json'

select_hotspots = {
    'B33': 'NE2',
    'B72': 'NE1,CD1,CD2,CG,CE3,CZ2,CZ3',
    'B92': 'CD2,CD1,CE1,CE2,CZ',
}
select_hbond_donor = {
    'B38': 'NH1,NH2',
}
select_hbond_acceptor = {
    'B90': 'OD1,OD2',
    'B94': 'OD1,OD2',
}

infer_ori_strategy = 'hotspots'
is_non_loopy       = True

rfd3_json = str(Path(configs_dir) / json_name)

payload = {
    design_name: {
        'dialect': dialect,
        'input': input_pdb,
        'contig': contig,
        'length': length,
        'redesign_motif_sidechains': redesign_motif_sidechains,
        'select_hotspots': select_hotspots,
        'select_hbond_donor': select_hbond_donor,
        'select_hbond_acceptor': select_hbond_acceptor,
        'infer_ori_strategy': infer_ori_strategy,
        'is_non_loopy': is_non_loopy,
    }
}

with open(rfd3_json, 'w') as f:
    json.dump(payload, f, indent=4)

print(f'Wrote JSON → {rfd3_json}')

### Run RFD3

> **Colab note**: This runs inference directly — no job scheduler. For a teaching demo, `diffusion_batch_size=1` and `n_batches=2` keeps runtime under ~10 min on a T4 GPU. Increase for a real campaign.

In [ ]:
RFD3_CKPT            = f'{MODELS_DIR}/rfd3_latest.ckpt'
diffusion_batch_size = 1   # increase for real campaign (was 2)
n_batches            = 2   # total designs = batch_size × n_batches

cmd = (
    f'rfd3 design'
    f' out_dir="{diffusion_out_dir}"'
    f' inputs="{rfd3_json}"'
    f' ckpt_path="{RFD3_CKPT}"'
    f' diffusion_batch_size={diffusion_batch_size}'
    f' n_batches={n_batches}'
    f' inference_sampler.step_scale=3'
    f' inference_sampler.gamma_0=0.2'
)

run_step(cmd, label='RFD3 diffusion')
print('\nRFD3 finished.')

### Process RFD3 outputs

In [ ]:
rfd3_metrics_csv = Path(scores_dir) / 'rfd3_metrics_with_json_path.csv'

json_paths = sorted(glob.glob(os.path.join(diffusion_out_dir, '*.json')))
if not json_paths:
    raise SystemExit(f'No JSON files found in {diffusion_out_dir!r} — run RFD3 first.')

rows = []
all_keys = set(['json_path'])

for jp in json_paths:
    with open(jp, 'r') as f:
        data = json.load(f)
    metrics = data.get('metrics', {})
    row = {'json_path': os.path.abspath(jp)}
    for k, v in metrics.items():
        if k in ('diffused_com', 'fixed_com') and isinstance(v, (list, tuple)) and len(v) == 3:
            row[f'{k}_x'], row[f'{k}_y'], row[f'{k}_z'] = v
            all_keys.update({f'{k}_x', f'{k}_y', f'{k}_z'})
        else:
            row[k] = v
            all_keys.add(k)
    rows.append(row)

fieldnames = ['json_path'] + sorted(k for k in all_keys if k != 'json_path')

with open(rfd3_metrics_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in rows:
        writer.writerow(r)

print(f'Wrote {len(rows)} rows → {rfd3_metrics_csv}')
print('Columns:', fieldnames)

In [ ]:
df = pd.read_csv(rfd3_metrics_csv)
df.head(10)

In [ ]:
metrics = [
    'alanine_content', 'glycine_content', 'helix_fraction', 'loop_fraction',
    'sheet_fraction', 'max_ca_deviation', 'n_chainbreaks',
    'n_clashing.interresidue_clashes_w_backbone',
    'n_clashing.interresidue_clashes_w_sidechain',
    'non_loop_fraction', 'radius_of_gyration',
]
metrics = [m for m in metrics if m in df.columns]

n_cols = 4
n_rows = math.ceil(len(metrics) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = axes.flatten()

def plot_hist(ax, series, title):
    s = series.dropna()
    if s.empty:
        ax.set_title(f'{title} (no data)'); ax.axis('off'); return
    ax.hist(s, bins=30)
    ax.set_title(title)
    ax.set_ylabel('count')

for ax, col in zip(axes, metrics):
    plot_hist(ax, df[col], col)
for ax in axes[len(metrics):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

### Filter designs (optional)

In [ ]:
df_filt = df[
    (df['loop_fraction'] < 0.4) &
    (df['non_loop_fraction'] > 0.6)
].copy()

print(f'Total designs  : {len(df)}')
print(f'Passing designs: {len(df_filt)}')
df_filt.head()

In [ ]:
filter_name  = 'passing_loop_nonloop'
filtered_dir = Path(diffusion_out_dir) / filter_name
filtered_dir.mkdir(parents=True, exist_ok=True)

def copy_design(json_path, dst_dir):
    jp = Path(json_path)
    if not jp.exists():
        return 0
    copied = 0
    dst_json = dst_dir / jp.name
    if not dst_json.exists():
        shutil.copy2(jp, dst_json); copied += 1
    for cif in [jp.with_name(jp.stem + '.cif'), jp.with_name(jp.stem + '.cif.gz')]:
        if cif.exists():
            dst_cif = dst_dir / cif.name
            if not dst_cif.exists():
                shutil.copy2(cif, dst_cif); copied += 1
    return copied

total_copied = missing = 0
for p in df_filt['json_path'].dropna().unique():
    if not os.path.exists(p):
        print(f'[MISSING] {p}'); missing += 1; continue
    total_copied += copy_design(p, filtered_dir)

print(f'Copied files: {total_copied} | Missing: {missing}')
print(f'Output folder: {filtered_dir}')

## MPNN

ProteinMPNN designs amino acid sequences onto each generated backbone. The binder chain (A) is redesigned freely.

In [ ]:
seed              = 42
batch_size        = 5
number_of_batches = 1
MPNN_CKPT         = f'{MODELS_DIR}/proteinmpnn_v_48_020.pt'
chains_to_design  = 'A'

### Run MPNN

> **Colab note**: Runs each structure sequentially instead of as an LSF array job.

In [ ]:
structures = sorted(glob.glob(f'{diffusion_out_dir}/*/*cif.gz'))
print(f'Structures to process: {len(structures)}')

for i, structure in enumerate(structures, 1):
    bn = os.path.basename(structure).replace('.cif.gz', '')
    this_outdir = f'{mpnn_out_dir}/{bn}'
    os.makedirs(this_outdir, exist_ok=True)

    cmd = (
        f'python {LIGANDMPNN_DIR}/run.py'
        f' --seed {seed}'
        f' --pdb_path "{structure}"'
        f' --out_folder "{this_outdir}"'
        f' --batch_size {batch_size}'
        f' --number_of_batches {number_of_batches}'
        f' --model_type protein_mpnn'
        f' --checkpoint_protein_mpnn "{MPNN_CKPT}"'
        f' --chains_to_design "{chains_to_design}"'
    )
    run_step(cmd, label=f'MPNN [{i}/{len(structures)}] {bn}')

print('\nAll MPNN jobs finished.')

## RF3

RosettaFold3 predicts the full complex structure for each MPNN-designed sequence — the validation step. High pLDDT and low PAE indicate a confident, well-packed design.

The target complex has two chains (F and G); the binder is chain A. We use an `rf3_template.json` that pre-specifies the target chains — only the binder sequence (chain A) changes per design.

### Generate RF3 JSONs

In [ ]:
mpnn_root = mpnn_out_dir

# rf3_template.json defines the fixed target chains (F, G).
# ⚠ Update the 'path' field inside the template to point to your Drive-hosted target CIF.
rf3_template_path = f'{INPUTS_DIR}/rf3_template.json'

rf3_json_dir = Path(configs_dir) / 'rf3'
rf3_json_dir.mkdir(parents=True, exist_ok=True)

MAX_R_PER_200   = None
ONLY_IDS_1_TO_9 = False

ID_RE = re.compile(r'(?:^|[, ])id=(\d+)(?:,|$)')

def parse_fasta(path):
    header = None; seq_chunks = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.startswith('>'):
                if header is not None:
                    yield header, ''.join(seq_chunks)
                header = line[1:].strip(); seq_chunks = []
            else:
                seq_chunks.append(line)
        if header is not None:
            yield header, ''.join(seq_chunks)

def extract_id(header):
    m = ID_RE.search(header)
    return int(m.group(1)) if m else None

def chain_a(seq): return seq.split(':', 1)[0]

def r_per_200(seq):
    L = len(seq)
    return 0.0 if L == 0 else (seq.count('R') / L) * 200.0

with open(rf3_template_path) as f:
    template = json.load(f)

if 'components' not in template or not isinstance(template['components'], list):
    raise ValueError("RF3 template must contain a top-level 'components' list.")

template_name = template.get('name', '')

binder_idx = None
for i, comp in enumerate(template['components']):
    if isinstance(comp, dict) and comp.get('chain_id') == 'A' and 'seq' in comp:
        binder_idx = i; break
if binder_idx is None:
    raise ValueError("Could not find binder component with {'chain_id': 'A', 'seq': ...} in template.")

fa_files = sorted(glob.glob(os.path.join(mpnn_root, '**', 'seqs', '*.cif.gz.fa'), recursive=True))

total_records = total_written = 0
skipped_no_id = skipped_fail = skipped_ids = 0

for fa_path in fa_files:
    model_dir = os.path.basename(os.path.dirname(os.path.dirname(fa_path)))
    for header, seq in parse_fasta(fa_path):
        total_records += 1
        seq_id = extract_id(header)
        if seq_id is None: skipped_no_id += 1; continue
        if ONLY_IDS_1_TO_9 and not (1 <= seq_id <= 9): skipped_ids += 1; continue
        a = chain_a(seq)
        if MAX_R_PER_200 is not None and r_per_200(a) > float(MAX_R_PER_200): skipped_fail += 1; continue
        j = deepcopy(template)
        j['components'][binder_idx]['seq'] = a
        safe_model = model_dir
        if template_name and safe_model.startswith(template_name):
            safe_model = safe_model[len(template_name):].lstrip('_')
        name = f'{safe_model}__id{seq_id}'
        j['name'] = name
        out_path = rf3_json_dir / f'{name}.json'
        with open(out_path, 'w') as out_f:
            json.dump(j, out_f, indent=4)
        total_written += 1

print(f'FASTA files scanned  : {len(fa_files)}')
print(f'Total FASTA records  : {total_records}')
print(f'Skipped (no id=)     : {skipped_no_id}')
print(f'Skipped (R filter)   : {skipped_fail}')
print(f'JSONs written        : {total_written}')
print(f'Output dir           : {rf3_json_dir}')

### Run RF3

> **Colab note**: Runs each design sequentially. `num_steps=20` for a quick demo; use 50–200 for production.

In [ ]:
# rf3_foundry_01_24_latest_remapped.ckpt is the checkpoint compatible with
# the rc-foundry rf3 CLI (different format from the standalone rf3 release).
RF3_CKPT  = f'{MODELS_DIR}/rf3_foundry_01_24_latest_remapped.ckpt'
num_steps = 20   # 50-200 for production

json_files = sorted(Path(rf3_json_dir).glob('*.json'))
if not json_files:
    raise RuntimeError(f'No JSON files found in {rf3_json_dir}')

print(f'Running RF3 on {len(json_files)} design(s)...')

for i, jf in enumerate(json_files, 1):
    out_dir = os.path.join(rf3_out_dir, jf.stem)
    cmd = (
        f'rf3 fold'
        f' inference_engine=rf3'
        f' inputs="{jf}"'
        f' out_dir="{out_dir}"'
        f' ckpt_path="{RF3_CKPT}"'
        f' num_steps={num_steps}'
        f' annotate_b_factor_with_plddt=True'
        f' early_stopping_plddt_threshold=0'
    )
    run_step(cmd, label=f'RF3 [{i}/{len(json_files)}] {jf.stem}')

print('\nAll RF3 jobs finished.')

## Filter and plot RF3 outputs

Key metrics for binder design:
- **`best_binder_plddt`**: confidence of the binder fold (0–1); aim for > 0.7
- **`AF_best_min_pae` / `AG_best_min_pae`**: interface PAE between binder and each target chain (lower = better)
- **`ipsae_interface`**: interface-specific PAE; low values indicate a well-defined binding pose

In [ ]:
from rf3_metrics_foundry import gather_rf3_metrics_foundry

# Chain order in the RF3 output matches the input JSON components:
#   index 0 = binder  (chain A, designed)
#   index 1 = CLR     (chain B from 6e3y_binding_site.pdb)
#   index 2 = RAMP1   (chain C from 6e3y_binding_site.pdb)
OUT_CSV = f'{scores_dir}/rf3_gathered_metrics.csv'

df_rf3 = gather_rf3_metrics_foundry(
    parent=rf3_out_dir,
    binder_chain_idx=0,
    target_chain_idxs=[1, 2],
    out_csv=OUT_CSV,
)
df_rf3.head()

In [ ]:
df = df_rf3.copy()

# ── Scatter: binder pLDDT vs interface PAE (both target chains) ───────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

pae_cols = [c for c in df.columns if c.startswith('pae_binder_vs_chain')]
labels   = ['Binder–CLR (chain B)', 'Binder–RAMP1 (chain C)']
colors   = df.get('iptm', pd.Series(dtype=float)).to_numpy(float)

for ax, col, label in zip(axes, pae_cols, labels):
    x = df[col].to_numpy(float)
    y = df['binder_plddt'].to_numpy(float)
    c = colors if np.any(np.isfinite(colors)) else 'steelblue'
    mask = np.isfinite(x) & np.isfinite(y)
    sc = ax.scatter(x[mask], y[mask],
                    c=c[mask] if isinstance(c, np.ndarray) else c,
                    s=40, alpha=0.85, linewidths=0, cmap='viridis')
    ax.set_title(label)
    ax.set_xlabel('min interface PAE (Å)')
    ax.set_ylabel('binder pLDDT')
    ax.set_xlim(left=0); ax.set_ylim(0, 1)
    ax.grid(True, linestyle=':', linewidth=0.6, alpha=0.6)
    if isinstance(c, np.ndarray):
        fig.colorbar(sc, ax=ax).set_label('iptm')

fig.suptitle('RF3 binder quality — CLR:RAMP1 campaign', y=1.02)
fig.tight_layout()
plt.show()

# ── Ranked table ─────────────────────────────────────────────────────────────
print(df.sort_values('ranking_score', ascending=False)
        [['design_id', 'binder_plddt', 'iptm', 'ranking_score'] + pae_cols]
        .to_string(index=False))